In [4]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')

filepath = r'C:\Users\SujitR\Desktop\rfai\phone_data_5g.csv'

df = pd.read_csv(filepath)

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
print(df.head(3).to_string())
print(f"\nNull counts:")
print(df.isnull().sum())
print(f"\nBasic stats:")
print(df.describe().round(2))

Shape: (684218, 20)

Columns: ['measurement_id', 'time', 'latitude', 'longitude', 'beam_index', 'beam_type', 'rsrp_dbm', 'rsrq_db', 'rssi_dbm', 'sinr_db', 'pathloss_db', 'dl_throughput_mbps', 'ul_throughput_mbps', 'timing_advance', 'operator', 'frequency_khz', 'channel_number', 'pci', 'cell_id_dummy', 'gnb_id_dummy']

First 3 rows:
   measurement_id                     time   latitude  longitude  beam_index      beam_type  rsrp_dbm  rsrq_db  rssi_dbm  sinr_db  pathloss_db  dl_throughput_mbps  ul_throughput_mbps  timing_advance operator  frequency_khz  channel_number    pci  cell_id_dummy  gnb_id_dummy
0               0  2024-10-13 15:00:41.418  48.197830  16.370186         0.0   Serving beam     -72.5    -10.4     -40.8     25.1         91.0          186.399994                 0.1             NaN        A      2125450.0        425090.0  354.0            NaN           NaN
1               1  2024-10-13 15:22:42.371  48.201199  16.343235         0.0  Detected beam    -120.1    -14.6      

In [5]:
cell_filepath =r'C:\Users\SujitR\Desktop\rfai\cell_info_final_5g.csv'  # update path

df_cells = pd.read_csv(cell_filepath)

print(f"Cell sheet shape: {df_cells.shape}")
print(f"Columns: {list(df_cells.columns)}")
print(f"\nFirst 5 rows:")
print(df_cells.head(5).to_string())

Cell sheet shape: (35, 10)
Columns: ['operator', 'technology', 'gnb_id_dummy', 'cell_id_dummy', 'pci', 'channel_number', 'longitude', 'latitude', 'height_m', 'azimuth_deg']

First 5 rows:
  operator technology  gnb_id_dummy  cell_id_dummy  pci  channel_number  longitude   latitude  height_m  azimuth_deg
0        A       5GNR             1              1  114          636000  16.370170  48.185332      29.0         35.0
1        A       5GNR            16             25  370          636000  16.373665  48.187470      27.0        230.0
2        A       5GNR            11             19  286          636000  16.367520  48.190033      58.0        155.0
3        A       5GNR            16             26  371          636000  16.373665  48.187470      27.0        190.0
4        A       5GNR             2              2  116          636000  16.371823  48.184170      37.0        270.0


In [6]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
matplotlib.use('Agg')

print("Imports good — starting Project 3: Coverage Predictor")
print("Dataset: 684,218 NR measurements across 35 cells")
print("Features: GPS + signal + beam + site geometry")
print("Target: Predict NR RSRP at unmeasured locations")

Imports good — starting Project 3: Coverage Predictor
Dataset: 684,218 NR measurements across 35 cells
Features: GPS + signal + beam + site geometry
Target: Predict NR RSRP at unmeasured locations


In [7]:
# ── Load both files ───────────────────────────────────────
print("Loading datasets...")

# Update these paths
meas_filepath = r'C:\Users\SujitR\Desktop\rfai\phone_data_5g.csv'
cell_filepath  = r'C:\Users\SujitR\Desktop\rfai\cell_info_final_5g.csv'

df_meas  = pd.read_csv(meas_filepath)
df_cells = pd.read_csv(cell_filepath)

print(f"Measurements loaded: {df_meas.shape}")
print(f"Cell info loaded:    {df_cells.shape}")

# ── Separate serving and neighbor beams ───────────────────
print("\nSeparating serving and neighbor beams...")

df_serving  = df_meas[df_meas['beam_type'] == 'Serving beam'].copy()
df_neighbor = df_meas[df_meas['beam_type'] == 'Detected beam'].copy()

print(f"Serving beam measurements:  {len(df_serving):,}")
print(f"Neighbor beam measurements: {len(df_neighbor):,}")

# ── Merge serving beams with cell info on PCI ─────────────
print("\nMerging serving beams with cell site information...")

df_serving = df_serving.merge(
    df_cells[['pci', 'longitude', 'latitude',
              'height_m', 'azimuth_deg',
              'cell_id_dummy', 'gnb_id_dummy']],
    on='pci',
    how='left',
    suffixes=('_meas', '_site')
)

print(f"After merge shape: {df_serving.shape}")
print(f"Merge success rate: "
      f"{df_serving['height_m'].notna().sum() / len(df_serving) * 100:.1f}%")

# ── Compute propagation features ──────────────────────────
print("\nComputing propagation geometry features...")

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Compute distance in meters between two GPS points
    using the Haversine formula
    Same formula used in drive test tools
    """
    R = 6371000  # Earth radius in meters

    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (np.sin(dlat/2)**2 +
         np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2)
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

def compute_bearing(lat1, lon1, lat2, lon2):
    """
    Compute bearing in degrees from point 1 to point 2
    0° = North, 90° = East, 180° = South, 270° = West
    """
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)

    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = (np.cos(lat1) * np.sin(lat2) -
         np.sin(lat1) * np.cos(lat2) * np.cos(dlon))

    bearing = np.degrees(np.arctan2(x, y))
    return (bearing + 360) % 360  # normalize to 0-360

# Distance from measurement point to serving site
df_serving['distance_to_site_m'] = haversine_distance(
    df_serving['latitude_meas'],  df_serving['longitude_meas'],
    df_serving['latitude_site'],  df_serving['longitude_site']
)

# Bearing from site to UE measurement point
df_serving['bearing_to_ue'] = compute_bearing(
    df_serving['latitude_site'],  df_serving['longitude_site'],
    df_serving['latitude_meas'],  df_serving['longitude_meas']
)

# Angle off boresight — how far UE is from antenna main beam
# 0° = UE directly in front of antenna = best signal
# 90° = UE off the side = weaker
# 180° = UE behind antenna = worst
df_serving['angle_off_boresight'] = np.abs(
    ((df_serving['bearing_to_ue'] -
      df_serving['azimuth_deg'] + 180) % 360) - 180
)

# Frequency band flag — 2.1GHz vs 3.5GHz
df_serving['freq_band'] = np.where(
    df_serving['frequency_khz'] < 3000000, 'n1_2100', 'n78_3500'
)
df_serving['freq_band_num'] = np.where(
    df_serving['frequency_khz'] < 3000000, 2100, 3500
)

# Parse timestamp features
df_serving['time'] = pd.to_datetime(df_serving['time'])
df_serving['hour']       = df_serving['time'].dt.hour
df_serving['day_of_week'] = df_serving['time'].dt.dayofweek
df_serving['is_peak']    = df_serving['hour'].between(8, 20).astype(int)

print(f"Distance range: "
      f"{df_serving['distance_to_site_m'].min():.0f}m to "
      f"{df_serving['distance_to_site_m'].max():.0f}m")
print(f"Angle off boresight range: "
      f"{df_serving['angle_off_boresight'].min():.1f}° to "
      f"{df_serving['angle_off_boresight'].max():.1f}°")

# ── Add neighbor features ──────────────────────────────────
print("\nEngineering neighbor beam features per measurement point...")

# For each measurement point get best neighbor RSRP
# and count of strong neighbors
neighbor_stats = df_neighbor.groupby('measurement_id').agg(
    best_neighbor_rsrp  = ('rsrp_dbm', 'max'),
    worst_neighbor_rsrp = ('rsrp_dbm', 'min'),
    mean_neighbor_rsrp  = ('rsrp_dbm', 'mean'),
    neighbor_count      = ('rsrp_dbm', 'count'),
    best_neighbor_rsrq  = ('rsrq_db', 'max'),
).reset_index()

# Join neighbor stats back to serving measurements
df_serving = df_serving.merge(
    neighbor_stats, on='measurement_id', how='left'
)

# Interference indicator — how close is best neighbor to serving
df_serving['neighbor_rsrp_delta'] = (
    df_serving['best_neighbor_rsrp'] - df_serving['rsrp_dbm']
)

print(f"Neighbor features added")
print(f"Measurements with neighbor info: "
      f"{df_serving['neighbor_count'].notna().sum():,}")

# ── Final cleaning ─────────────────────────────────────────
print("\nFinal cleaning...")

# Drop rows where target variable is missing
df_model = df_serving.dropna(subset=['rsrp_dbm']).copy()

# Fill neighbor nulls — some locations had no detected neighbors
df_model['neighbor_count']      = df_model['neighbor_count'].fillna(0)
df_model['best_neighbor_rsrp']  = df_model['best_neighbor_rsrp'].fillna(
    df_model['rsrp_dbm'] - 20)   # assume neighbors 20dB below serving
df_model['neighbor_rsrp_delta'] = df_model['neighbor_rsrp_delta'].fillna(-20)
df_model['mean_neighbor_rsrp']  = df_model['mean_neighbor_rsrp'].fillna(
    df_model['rsrp_dbm'] - 25)

print(f"\nFinal dataset shape: {df_model.shape}")
print(f"\nKey feature summary:")
summary_cols = ['rsrp_dbm', 'rsrq_db', 'sinr_db',
                'distance_to_site_m', 'angle_off_boresight',
                'height_m', 'pathloss_db', 'neighbor_count']
print(df_model[summary_cols].describe().round(2))

print(f"\nFrequency band distribution:")
print(df_model['freq_band'].value_counts())

print(f"\nMeasurements per cell (top 10):")
print(df_model['pci'].value_counts().head(10))

Loading datasets...
Measurements loaded: (684218, 20)
Cell info loaded:    (35, 10)

Separating serving and neighbor beams...
Serving beam measurements:  200,129
Neighbor beam measurements: 484,069

Merging serving beams with cell site information...
After merge shape: (207858, 26)
Merge success rate: 18.3%

Computing propagation geometry features...
Distance range: 10m to 8861m
Angle off boresight range: 0.0° to 169.4°

Engineering neighbor beam features per measurement point...
Neighbor features added
Measurements with neighbor info: 154,378

Final cleaning...

Final dataset shape: (207858, 40)

Key feature summary:
        rsrp_dbm    rsrq_db    sinr_db  distance_to_site_m  \
count  207858.00  207858.00  207804.00            37999.00   
mean      -93.17     -11.48      12.52              957.04   
std        11.06       1.66       8.97             1750.43   
min      -138.10     -39.10     -28.60                9.78   
25%      -101.10     -11.90       5.80              232.88   
50

In [8]:
# Check PCI overlap between measurements and cell info
print("=== PCI Mismatch Diagnosis ===")

meas_pcis = set(df_serving['pci'].dropna().unique())
cell_pcis = set(df_cells['pci'].unique())

print(f"Unique PCIs in measurements: {len(meas_pcis)}")
print(f"Unique PCIs in cell info:    {len(cell_pcis)}")
print(f"PCIs in both:                {len(meas_pcis & cell_pcis)}")
print(f"PCIs in measurements only:   {len(meas_pcis - cell_pcis)}")
print(f"PCIs in cell info only:      {len(cell_pcis - meas_pcis)}")

print(f"\nPCIs in measurements (sample):")
print(sorted(list(meas_pcis))[:20])

print(f"\nPCIs in cell info:")
print(sorted(list(cell_pcis)))

print(f"\nChannel numbers in measurements (unique):")
print(sorted(df_serving['channel_number'].dropna().unique()))

print(f"\nChannel numbers in cell info:")
print(sorted(df_cells['channel_number'].unique()))

=== PCI Mismatch Diagnosis ===
Unique PCIs in measurements: 278
Unique PCIs in cell info:    30
PCIs in both:                30
PCIs in measurements only:   248
PCIs in cell info only:      0

PCIs in measurements (sample):
[np.float64(0.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(17.0), np.float64(19.0), np.float64(20.0), np.float64(23.0), np.float64(24.0), np.float64(25.0), np.float64(26.0), np.float64(27.0), np.float64(28.0), np.float64(29.0), np.float64(33.0), np.float64(34.0), np.float64(35.0), np.float64(37.0), np.float64(38.0), np.float64(39.0)]

PCIs in cell info:
[np.int64(87), np.int64(114), np.int64(116), np.int64(136), np.int64(138), np.int64(140), np.int64(146), np.int64(156), np.int64(157), np.int64(158), np.int64(201), np.int64(221), np.int64(238), np.int64(239), np.int64(259), np.int64(285), np.int64(286), np.int64(320), np.int64(331), np.int64(338), np.int64(369), np.int64(370), np.int64(371), np.int64(390), np.int64(419), np.int64(423), np.int64(

In [11]:
# Load LTE measurement data
lte_filepath = r'C:\Users\SujitR\Desktop\rfai\phone_data_lte.csv'

df_lte = pd.read_csv(lte_filepath)

print(f"Shape: {df_lte.shape}")
print(f"\nColumns: {list(df_lte.columns)}")
print(f"\nFirst 3 rows:")
print(df_lte.head(3).to_string())
print(f"\nNull counts:")
print(df_lte.isnull().sum())
print(f"\nBasic stats:")
print(df_lte.describe().round(2))

Shape: (1183683, 19)

Columns: ['measurement_id', 'time', 'latitude', 'longitude', 'rsrp_dbm', 'rsrq_db', 'rssi_dbm', 'sinr_db', 'pathloss_db', 'dl_throughput_mbps', 'ul_throughput_mbps', 'operator', 'timing_advance', 'frequency_khz', 'channel_number', 'pci', 'cell_id', 'enb_id', 'sector_id']

First 3 rows:
   measurement_id                     time   latitude  longitude  rsrp_dbm  rsrq_db  rssi_dbm  sinr_db  pathloss_db  dl_throughput_mbps  ul_throughput_mbps operator  timing_advance  frequency_khz  channel_number  pci  cell_id  enb_id  sector_id
0               0  2024-10-13 15:12:30.716  48.206005  16.340046    -104.4    -16.2     -68.3     -1.4        120.0                16.1                 0.1        A             NaN      2630000.0            2850  410   107011     418          3
1               1  2024-10-13 15:12:31.217  48.206009  16.339991    -103.5    -15.1     -68.3      0.2        119.0                13.5                 0.1        A             NaN      2630000.0      

In [12]:
# Load LTE cell info
lte_cell_filepath = r'C:\Users\SujitR\Desktop\rfai\cell_info_final_lte.csv'

df_lte_cells = pd.read_csv(lte_cell_filepath)

print(f"\nCell info shape: {df_lte_cells.shape}")
print(f"Columns: {list(df_lte_cells.columns)}")
print(df_lte_cells.head(5).to_string())


Cell info shape: (1597, 11)
Columns: ['operator', 'technology', 'enb_id', 'sector_id', 'cell_id', 'pci', 'channel_number', 'longitude', 'latitude', 'height_m', 'azimuth_deg']
  operator technology  enb_id  sector_id  cell_id  pci  channel_number  longitude   latitude  height_m  azimuth_deg
0        A        LTE     624         12   159756   37            1700  16.368760  48.202033        32        195.0
1        A        LTE    3534         21   904725  285            1871  16.367520  48.190033        58         20.0
2        A        LTE    3534         11   904715  285            1700  16.367520  48.190033        58         20.0
3        A        LTE    3534          6   904710  285            6250  16.367520  48.190033        58         20.0
4        A        LTE     471         13   120589  215            1700  16.386906  48.193860        34        265.0


In [13]:
# Quick PCI and cell_id overlap check
print("=== LTE Dataset Overlap Check ===")

meas_cellids = set(df_lte['cell_id'].dropna().unique())
cell_cellids = set(df_lte_cells['cell_id'].unique())

print(f"Unique cell_ids in measurements: {len(meas_cellids)}")
print(f"Unique cell_ids in cell info:    {len(cell_cellids)}")
print(f"cell_ids in both:                {len(meas_cellids & cell_cellids)}")
print(f"cell_ids in measurements only:   {len(meas_cellids - cell_cellids)}")
print(f"Match rate: "
      f"{len(meas_cellids & cell_cellids)/len(meas_cellids)*100:.1f}%")

print(f"\nOperator distribution in measurements:")
print(df_lte['operator'].value_counts())

print(f"\nFrequency bands in measurements:")
print(df_lte['frequency_khz'].value_counts().head(10))

print(f"\nFrequency bands in cell info:")
print(df_lte_cells['channel_number'].value_counts().head(10))

print(f"\nSample cell_ids in measurements:")
print(sorted(list(meas_cellids))[:10])

print(f"\nSample cell_ids in cell info:")
print(sorted(list(cell_cellids))[:10])

=== LTE Dataset Overlap Check ===
Unique cell_ids in measurements: 2952
Unique cell_ids in cell info:    1579
cell_ids in both:                903
cell_ids in measurements only:   2049
Match rate: 30.6%

Operator distribution in measurements:
operator
A    587728
C    302931
B    292746
Name: count, dtype: int64

Frequency bands in measurements:
frequency_khz
1855000.0    387688
1837500.0    267041
1815000.0    206170
801000.0      94894
2630000.0     78647
2650000.0     74587
1872100.0     25183
2162500.0     15997
2142500.0     12161
2680000.0      9673
Name: count, dtype: int64

Frequency bands in cell info:
channel_number
1700    154
525     150
1871    147
1300    146
6250    144
2850    135
325     129
3500    128
3050    118
6400    109
Name: count, dtype: int64

Sample cell_ids in measurements:
[np.int64(69379), np.int64(69384), np.int64(69389), np.int64(69399), np.int64(70406), np.int64(70407), np.int64(70411), np.int64(72195), np.int64(72203), np.int64(72204)]

Sample cell_id

In [14]:
# Filter to operator A only and recheck
print("=== Operator A Only Analysis ===")

df_lte_A = df_lte[df_lte['operator'] == 'A'].copy()
print(f"Operator A measurements: {len(df_lte_A):,}")

meas_A_cellids = set(df_lte_A['cell_id'].dropna().unique())
cell_cellids   = set(df_lte_cells['cell_id'].unique())

print(f"\nUnique cell_ids in operator A measurements: {len(meas_A_cellids)}")
print(f"Unique cell_ids in cell info:               {len(cell_cellids)}")
print(f"cell_ids in both:                           {len(meas_A_cellids & cell_cellids)}")
print(f"Match rate for operator A:                  "
      f"{len(meas_A_cellids & cell_cellids)/len(meas_A_cellids)*100:.1f}%")

print(f"\nOperator A frequency bands:")
print(df_lte_A['frequency_khz'].value_counts())

print(f"\nSample matched cell_ids:")
matched = list(meas_A_cellids & cell_cellids)[:5]
print(matched)
print(df_lte_cells[df_lte_cells['cell_id'].isin(matched)][
    ['cell_id', 'pci', 'longitude', 'latitude', 
     'height_m', 'azimuth_deg']].to_string())

=== Operator A Only Analysis ===
Operator A measurements: 587,728

Unique cell_ids in operator A measurements: 1154
Unique cell_ids in cell info:               1579
cell_ids in both:                           411
Match rate for operator A:                  35.6%

Operator A frequency bands:
frequency_khz
1855000.0    387688
801000.0      94894
2630000.0     78647
1872100.0     25183
Name: count, dtype: int64

Sample matched cell_ids:
[np.int64(133121), np.int64(133122), np.int64(159746), np.int64(299011), np.int64(106497)]
     cell_id  pci  longitude   latitude  height_m  azimuth_deg
391   133121  135  16.346526  48.189308        31         65.0
392   133122  136  16.346526  48.189308        31        105.0
469   106497  252  16.325982  48.199914        37         60.0
499   299011   53  16.323610  48.174430        23        300.0
548   159746   37  16.368760  48.202033        32        195.0


In [15]:
# Load the complete cell info file with all operators
all_cell_filepath= r'C:\Users\SujitR\Desktop\rfai\cell_info_final_lte.csv'

df_all_cells = pd.read_csv(all_cell_filepath)

print(f"Shape: {df_all_cells.shape}")
print(f"Columns: {list(df_all_cells.columns)}")
print(f"\nOperator distribution:")
print(df_all_cells['operator'].value_counts())
print(f"\nFirst 5 rows:")
print(df_all_cells.head(5).to_string())

Shape: (1597, 11)
Columns: ['operator', 'technology', 'enb_id', 'sector_id', 'cell_id', 'pci', 'channel_number', 'longitude', 'latitude', 'height_m', 'azimuth_deg']

Operator distribution:
operator
C    630
A    580
B    387
Name: count, dtype: int64

First 5 rows:
  operator technology  enb_id  sector_id  cell_id  pci  channel_number  longitude   latitude  height_m  azimuth_deg
0        A        LTE     624         12   159756   37            1700  16.368760  48.202033        32        195.0
1        A        LTE    3534         21   904725  285            1871  16.367520  48.190033        58         20.0
2        A        LTE    3534         11   904715  285            1700  16.367520  48.190033        58         20.0
3        A        LTE    3534          6   904710  285            6250  16.367520  48.190033        58         20.0
4        A        LTE     471         13   120589  215            1700  16.386906  48.193860        34        265.0


In [16]:
# Check new match rate
meas_cellids     = set(df_lte['cell_id'].dropna().unique())
all_cell_cellids = set(df_all_cells['cell_id'].unique())

print(f"\n=== Updated Match Rate ===")
print(f"Unique cell_ids in measurements: {len(meas_cellids)}")
print(f"Unique cell_ids in cell info:    {len(all_cell_cellids)}")
print(f"cell_ids in both:                {len(meas_cellids & all_cell_cellids)}")
print(f"Match rate:                      "
      f"{len(meas_cellids & all_cell_cellids)/len(meas_cellids)*100:.1f}%")


=== Updated Match Rate ===
Unique cell_ids in measurements: 2952
Unique cell_ids in cell info:    1579
cell_ids in both:                903
Match rate:                      30.6%


In [17]:
# Deep dive into the mismatch
print("=== Deep Mismatch Analysis ===")

unmatched_cellids = meas_cellids - all_cell_cellids
matched_cellids   = meas_cellids & all_cell_cellids

# How many measurements are in matched vs unmatched cells
matched_meas   = df_lte[df_lte['cell_id'].isin(matched_cellids)]
unmatched_meas = df_lte[~df_lte['cell_id'].isin(matched_cellids)]

print(f"Matched cells:              {len(matched_cellids)} cells")
print(f"Unmatched cells:            {len(unmatched_cellids)} cells")
print(f"\nMatched measurements:       {len(matched_meas):,} ({len(matched_meas)/len(df_lte)*100:.1f}%)")
print(f"Unmatched measurements:     {len(unmatched_meas):,} ({len(unmatched_meas)/len(df_lte)*100:.1f}%)")

print(f"\nOperator breakdown — matched measurements:")
print(matched_meas['operator'].value_counts())

print(f"\nOperator breakdown — unmatched measurements:")
print(unmatched_meas['operator'].value_counts())

print(f"\nFrequency bands — matched:")
print(matched_meas['frequency_khz'].value_counts().head(5))

print(f"\nFrequency bands — unmatched:")
print(unmatched_meas['frequency_khz'].value_counts().head(5))

print(f"\nGeographic extent — matched:")
print(f"  Lat: {matched_meas['latitude'].min():.4f} to {matched_meas['latitude'].max():.4f}")
print(f"  Lon: {matched_meas['longitude'].min():.4f} to {matched_meas['longitude'].max():.4f}")

print(f"\nGeographic extent — unmatched:")
print(f"  Lat: {unmatched_meas['latitude'].min():.4f} to {unmatched_meas['latitude'].max():.4f}")
print(f"  Lon: {unmatched_meas['longitude'].min():.4f} to {unmatched_meas['longitude'].max():.4f}")

=== Deep Mismatch Analysis ===
Matched cells:              903 cells
Unmatched cells:            2049 cells

Matched measurements:       567,195 (47.9%)
Unmatched measurements:     616,488 (52.1%)

Operator breakdown — matched measurements:
operator
A    322863
C    152222
B     92110
Name: count, dtype: int64

Operator breakdown — unmatched measurements:
operator
A    264865
B    200636
C    150709
Name: count, dtype: int64

Frequency bands — matched:
frequency_khz
1855000.0    207239
1815000.0    106758
1837500.0     84531
2630000.0     52830
801000.0      48020
Name: count, dtype: int64

Frequency bands — unmatched:
frequency_khz
1837500.0    182510
1855000.0    180449
1815000.0     99412
801000.0      46874
2650000.0     40569
Name: count, dtype: int64

Geographic extent — matched:
  Lat: 48.1583 to 48.2262
  Lon: 16.2542 to 16.3982

Geographic extent — unmatched:
  Lat: 48.1559 to 48.2289
  Lon: 16.2540 to 16.3996


In [18]:
# Finalize working dataset — matched measurements only
print("=== Finalizing Working Dataset ===")

df_work = df_lte[df_lte['cell_id'].isin(matched_cellids)].copy()

print(f"Working dataset: {len(df_work):,} measurements")
print(f"Cells covered:   {df_work['cell_id'].nunique()}")
print(f"Operators:       {df_work['operator'].nunique()}")
print(f"Date range:      {pd.to_datetime(df_work['time']).min()} "
      f"to {pd.to_datetime(df_work['time']).max()}")

# Merge with cell info
df_work = df_work.merge(
    df_all_cells[['cell_id', 'longitude', 'latitude',
                  'height_m', 'azimuth_deg',
                  'enb_id', 'sector_id']],
    on='cell_id',
    how='left',
    suffixes=('_meas', '_site')
)

print(f"\nAfter merge: {df_work.shape}")
print(f"Merge success: "
      f"{df_work['height_m'].notna().sum()/len(df_work)*100:.1f}%")

print(f"\nKey columns sample:")
print(df_work[['latitude_meas', 'longitude_meas',
               'latitude_site', 'longitude_site',
               'rsrp_dbm', 'rsrq_db', 'sinr_db',
               'height_m', 'azimuth_deg']].head(3).to_string())

=== Finalizing Working Dataset ===
Working dataset: 567,195 measurements
Cells covered:   903
Operators:       3
Date range:      2024-10-13 15:02:52.070000 to 2025-12-03 23:59:53.497000

After merge: (570522, 25)
Merge success: 100.0%

Key columns sample:
   latitude_meas  longitude_meas  latitude_site  longitude_site  rsrp_dbm  rsrq_db  sinr_db  height_m  azimuth_deg
0      48.206036       16.338327      48.202255       16.337912    -105.4    -19.9      0.0        32        345.0
1      48.206005       16.338322      48.202255       16.337912    -102.5    -17.8      5.9        32        345.0
2      48.206005       16.338322      48.202255       16.337912    -103.8    -18.2      5.9        32        345.0


In [19]:
print("=== Feature Engineering ===")
print("Computing propagation geometry features...")

# ── Deduplicate ───────────────────────────────────────────
# Some cell_ids matched multiple cell info rows
# Keep first match per measurement
df_work = df_work.drop_duplicates(subset=['measurement_id']).copy()
print(f"After deduplication: {len(df_work):,} rows")

# ── Haversine distance ────────────────────────────────────
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (np.sin(dlat/2)**2 +
         np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2)
    return R * 2 * np.arcsin(np.sqrt(a))

def compute_bearing(lat1, lon1, lat2, lon2):
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = (np.cos(lat1) * np.sin(lat2) -
         np.sin(lat1) * np.cos(lat2) * np.cos(dlon))
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

# ── Geometry features ─────────────────────────────────────
df_work['distance_m'] = haversine_distance(
    df_work['latitude_meas'],  df_work['longitude_meas'],
    df_work['latitude_site'],  df_work['longitude_site']
)

df_work['bearing_to_ue'] = compute_bearing(
    df_work['latitude_site'],  df_work['longitude_site'],
    df_work['latitude_meas'],  df_work['longitude_meas']
)

df_work['angle_off_boresight'] = np.abs(
    ((df_work['bearing_to_ue'] -
      df_work['azimuth_deg'] + 180) % 360) - 180
)

# Log distance — propagation follows log-distance path loss model
# This linearizes the relationship between distance and RSRP
df_work['log_distance'] = np.log10(
    df_work['distance_m'].clip(lower=10)
)

# ── Frequency features ────────────────────────────────────
# Higher frequency = more path loss = weaker signal
freq_bins = [0, 1000000, 1500000, 2000000, 2500000, 3000000, 99999999]
freq_labels = ['<1GHz', '1-1.5GHz', '1.5-2GHz',
               '2-2.5GHz', '2.5-3GHz', '>3GHz']

df_work['freq_band'] = pd.cut(
    df_work['frequency_khz'],
    bins=freq_bins,
    labels=freq_labels
)
df_work['freq_ghz'] = df_work['frequency_khz'] / 1e6

# ── Time features ─────────────────────────────────────────
df_work['time'] = pd.to_datetime(df_work['time'])
df_work['hour']        = df_work['time'].dt.hour
df_work['day_of_week'] = df_work['time'].dt.dayofweek
df_work['is_peak']     = df_work['hour'].between(8, 20).astype(int)

# ── Operator encoding ─────────────────────────────────────
operator_map = {'A': 0, 'B': 1, 'C': 2}
df_work['operator_num'] = df_work['operator'].map(operator_map)

# ── Derived signal features ───────────────────────────────
# Interference ratio — RSSI includes all interference
# Higher RSSI relative to RSRP = more interference
df_work['interference_ratio'] = (
    df_work['rssi_dbm'] - df_work['rsrp_dbm']
)

# Coverage quality flag
df_work['poor_coverage'] = (df_work['rsrp_dbm'] < -100).astype(int)
df_work['good_coverage'] = (df_work['rsrp_dbm'] > -85).astype(int)

# ── Summary ───────────────────────────────────────────────
print(f"\nGeometry feature summary:")
geo_cols = ['distance_m', 'log_distance',
            'angle_off_boresight', 'height_m']
print(df_work[geo_cols].describe().round(2))

print(f"\nFrequency band distribution:")
print(df_work['freq_band'].value_counts())

print(f"\nCoverage quality distribution:")
total = len(df_work)
poor  = df_work['poor_coverage'].sum()
good  = df_work['good_coverage'].sum()
mid   = total - poor - good
print(f"  Good coverage (> -85 dBm):  {good:>7,} ({good/total*100:.1f}%)")
print(f"  Mid coverage:               {mid:>7,} ({mid/total*100:.1f}%)")
print(f"  Poor coverage (< -100 dBm): {poor:>7,} ({poor/total*100:.1f}%)")

print(f"\nDistance distribution:")
print(f"  Min:    {df_work['distance_m'].min():.0f}m")
print(f"  Median: {df_work['distance_m'].median():.0f}m")
print(f"  Max:    {df_work['distance_m'].max():.0f}m")
print(f"  Mean:   {df_work['distance_m'].mean():.0f}m")

print(f"\nAngle off boresight:")
print(f"  Mean:   {df_work['angle_off_boresight'].mean():.1f}°")
print(f"  Median: {df_work['angle_off_boresight'].median():.1f}°")
print(f"  Max:    {df_work['angle_off_boresight'].max():.1f}°")

print(f"\nFinal feature-engineered dataset: {df_work.shape}")

=== Feature Engineering ===
Computing propagation geometry features...
After deduplication: 567,195 rows

Geometry feature summary:
       distance_m  log_distance  angle_off_boresight   height_m
count   567195.00     567195.00            567129.00  567195.00
mean       316.07          2.39                32.94      28.25
std        243.41          0.32                33.20       7.30
min          1.32          1.00                 0.00       6.00
25%        158.31          2.20                 8.79      24.00
50%        261.06          2.42                22.72      28.00
75%        401.74          2.60                46.76      31.00
max       4546.16          3.66               180.00      58.00

Frequency band distribution:
freq_band
1.5-2GHz    412844
2.5-3GHz     91041
<1GHz        51768
2-2.5GHz     10895
1-1.5GHz         0
>3GHz            0
Name: count, dtype: int64

Coverage quality distribution:
  Good coverage (> -85 dBm):  202,187 (35.6%)
  Mid coverage:               272,

In [20]:
print("=== Feature Engineering ===")
print("Computing propagation geometry features...")

# ── Deduplicate ───────────────────────────────────────────
# Some cell_ids matched multiple cell info rows
# Keep first match per measurement
df_work = df_work.drop_duplicates(subset=['measurement_id']).copy()
print(f"After deduplication: {len(df_work):,} rows")

# ── Haversine distance ────────────────────────────────────
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (np.sin(dlat/2)**2 +
         np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2)
    return R * 2 * np.arcsin(np.sqrt(a))

def compute_bearing(lat1, lon1, lat2, lon2):
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = (np.cos(lat1) * np.sin(lat2) -
         np.sin(lat1) * np.cos(lat2) * np.cos(dlon))
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

# ── Geometry features ─────────────────────────────────────
df_work['distance_m'] = haversine_distance(
    df_work['latitude_meas'],  df_work['longitude_meas'],
    df_work['latitude_site'],  df_work['longitude_site']
)

df_work['bearing_to_ue'] = compute_bearing(
    df_work['latitude_site'],  df_work['longitude_site'],
    df_work['latitude_meas'],  df_work['longitude_meas']
)

df_work['angle_off_boresight'] = np.abs(
    ((df_work['bearing_to_ue'] -
      df_work['azimuth_deg'] + 180) % 360) - 180
)

# Log distance — propagation follows log-distance path loss model
# This linearizes the relationship between distance and RSRP
df_work['log_distance'] = np.log10(
    df_work['distance_m'].clip(lower=10)
)

# ── Frequency features ────────────────────────────────────
# Higher frequency = more path loss = weaker signal
freq_bins = [0, 1000000, 1500000, 2000000, 2500000, 3000000, 99999999]
freq_labels = ['<1GHz', '1-1.5GHz', '1.5-2GHz',
               '2-2.5GHz', '2.5-3GHz', '>3GHz']

df_work['freq_band'] = pd.cut(
    df_work['frequency_khz'],
    bins=freq_bins,
    labels=freq_labels
)
df_work['freq_ghz'] = df_work['frequency_khz'] / 1e6

# ── Time features ─────────────────────────────────────────
df_work['time'] = pd.to_datetime(df_work['time'])
df_work['hour']        = df_work['time'].dt.hour
df_work['day_of_week'] = df_work['time'].dt.dayofweek
df_work['is_peak']     = df_work['hour'].between(8, 20).astype(int)

# ── Operator encoding ─────────────────────────────────────
operator_map = {'A': 0, 'B': 1, 'C': 2}
df_work['operator_num'] = df_work['operator'].map(operator_map)

# ── Derived signal features ───────────────────────────────
# Interference ratio — RSSI includes all interference
# Higher RSSI relative to RSRP = more interference
df_work['interference_ratio'] = (
    df_work['rssi_dbm'] - df_work['rsrp_dbm']
)

# Coverage quality flag
df_work['poor_coverage'] = (df_work['rsrp_dbm'] < -100).astype(int)
df_work['good_coverage'] = (df_work['rsrp_dbm'] > -85).astype(int)

# ── Summary ───────────────────────────────────────────────
print(f"\nGeometry feature summary:")
geo_cols = ['distance_m', 'log_distance',
            'angle_off_boresight', 'height_m']
print(df_work[geo_cols].describe().round(2))

print(f"\nFrequency band distribution:")
print(df_work['freq_band'].value_counts())

print(f"\nCoverage quality distribution:")
total = len(df_work)
poor  = df_work['poor_coverage'].sum()
good  = df_work['good_coverage'].sum()
mid   = total - poor - good
print(f"  Good coverage (> -85 dBm):  {good:>7,} ({good/total*100:.1f}%)")
print(f"  Mid coverage:               {mid:>7,} ({mid/total*100:.1f}%)")
print(f"  Poor coverage (< -100 dBm): {poor:>7,} ({poor/total*100:.1f}%)")

print(f"\nDistance distribution:")
print(f"  Min:    {df_work['distance_m'].min():.0f}m")
print(f"  Median: {df_work['distance_m'].median():.0f}m")
print(f"  Max:    {df_work['distance_m'].max():.0f}m")
print(f"  Mean:   {df_work['distance_m'].mean():.0f}m")

print(f"\nAngle off boresight:")
print(f"  Mean:   {df_work['angle_off_boresight'].mean():.1f}°")
print(f"  Median: {df_work['angle_off_boresight'].median():.1f}°")
print(f"  Max:    {df_work['angle_off_boresight'].max():.1f}°")

print(f"\nFinal feature-engineered dataset: {df_work.shape}")

=== Feature Engineering ===
Computing propagation geometry features...
After deduplication: 567,195 rows

Geometry feature summary:
       distance_m  log_distance  angle_off_boresight   height_m
count   567195.00     567195.00            567129.00  567195.00
mean       316.07          2.39                32.94      28.25
std        243.41          0.32                33.20       7.30
min          1.32          1.00                 0.00       6.00
25%        158.31          2.20                 8.79      24.00
50%        261.06          2.42                22.72      28.00
75%        401.74          2.60                46.76      31.00
max       4546.16          3.66               180.00      58.00

Frequency band distribution:
freq_band
1.5-2GHz    412844
2.5-3GHz     91041
<1GHz        51768
2-2.5GHz     10895
1-1.5GHz         0
>3GHz            0
Name: count, dtype: int64

Coverage quality distribution:
  Good coverage (> -85 dBm):  202,187 (35.6%)
  Mid coverage:               272,

In [21]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('LTE Coverage Dataset — Feature Analysis',
             fontsize=14, fontweight='bold')

# ── Plot 1 — RSRP distribution ────────────────────────────
ax = axes[0, 0]
ax.hist(df_work['rsrp_dbm'], bins=80,
        color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(x=-85,  color='green',  linewidth=1.5,
           linestyle='--', label='Good > -85')
ax.axvline(x=-100, color='red',    linewidth=1.5,
           linestyle='--', label='Poor < -100')
ax.set_title('RSRP distribution — target variable',
             fontweight='bold')
ax.set_xlabel('RSRP (dBm)')
ax.set_ylabel('Count')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Plot 2 — RSRP vs distance ─────────────────────────────
ax = axes[0, 1]
# Sample 10,000 points for scatter — faster rendering
sample = df_work.sample(10000, random_state=42)
scatter = ax.scatter(
    sample['distance_m'],
    sample['rsrp_dbm'],
    c=sample['freq_ghz'],
    cmap='RdYlGn_r',
    alpha=0.3, s=1
)
plt.colorbar(scatter, ax=ax, label='Frequency (GHz)')
ax.set_title('RSRP vs distance — core relationship',
             fontweight='bold')
ax.set_xlabel('Distance to site (m)')
ax.set_ylabel('RSRP (dBm)')
ax.grid(True, alpha=0.3)

# ── Plot 3 — RSRP vs log distance ────────────────────────
ax = axes[0, 2]
ax.scatter(
    sample['log_distance'],
    sample['rsrp_dbm'],
    c=sample['freq_ghz'],
    cmap='RdYlGn_r',
    alpha=0.3, s=1
)
# Fit a line to show linearization
z = np.polyfit(
    df_work['log_distance'].dropna(),
    df_work['rsrp_dbm'].dropna(),
    1
)
p = np.poly1d(z)
x_line = np.linspace(1, 3.7, 100)
ax.plot(x_line, p(x_line), 'r-', linewidth=2,
        label=f'Fit: slope={z[0]:.1f} dB/decade')
ax.set_title('RSRP vs log(distance) — linearized',
             fontweight='bold')
ax.set_xlabel('Log10(distance)')
ax.set_ylabel('RSRP (dBm)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Plot 4 — RSRP vs angle off boresight ─────────────────
ax = axes[1, 0]
angle_bins = [0, 15, 30, 45, 60, 90, 120, 180]
df_work['angle_bin'] = pd.cut(
    df_work['angle_off_boresight'],
    bins=angle_bins
)
angle_rsrp = df_work.groupby(
    'angle_bin', observed=True
)['rsrp_dbm'].mean()
ax.bar(range(len(angle_rsrp)),
       angle_rsrp.values,
       color='coral', edgecolor='white', alpha=0.8)
ax.set_xticks(range(len(angle_rsrp)))
ax.set_xticklabels([str(b) for b in angle_rsrp.index],
                    rotation=45, fontsize=8)
ax.set_title('Mean RSRP by angle off boresight',
             fontweight='bold')
ax.set_xlabel('Angle off boresight (degrees)')
ax.set_ylabel('Mean RSRP (dBm)')
ax.grid(True, alpha=0.3, axis='y')

# ── Plot 5 — RSRP by frequency band ──────────────────────
ax = axes[1, 1]
band_order = ['<1GHz', '1-1.5GHz', '1.5-2GHz',
              '2-2.5GHz', '2.5-3GHz', '>3GHz']
band_rsrp = df_work.groupby(
    'freq_band', observed=True
)['rsrp_dbm'].mean().reindex(band_order)
colors_band = ['#2ecc71', '#27ae60', '#f39c12',
               '#e67e22', '#e74c3c', '#c0392b']
ax.bar(range(len(band_rsrp)),
       band_rsrp.values,
       color=colors_band[:len(band_rsrp)],
       edgecolor='white', alpha=0.8)
ax.set_xticks(range(len(band_rsrp)))
ax.set_xticklabels(band_rsrp.index, rotation=30, fontsize=8)
ax.set_title('Mean RSRP by frequency band',
             fontweight='bold')
ax.set_xlabel('Frequency band')
ax.set_ylabel('Mean RSRP (dBm)')
ax.grid(True, alpha=0.3, axis='y')

# ── Plot 6 — RSRP by operator ─────────────────────────────
ax = axes[1, 2]
op_colors = {'A': '#3498db', 'B': '#e74c3c', 'C': '#2ecc71'}
for op in ['A', 'B', 'C']:
    subset = df_work[df_work['operator'] == op]['rsrp_dbm']
    if len(subset) > 0:
        ax.hist(subset, bins=60, alpha=0.5,
                color=op_colors[op],
                label=f'Operator {op} (n={len(subset):,})')
ax.axvline(x=-85,  color='black', linewidth=1,
           linestyle='--', alpha=0.5)
ax.axvline(x=-100, color='black', linewidth=1,
           linestyle='--', alpha=0.5)
ax.set_title('RSRP distribution by operator',
             fontweight='bold')
ax.set_xlabel('RSRP (dBm)')
ax.set_ylabel('Count')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('coverage_feature_analysis.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: coverage_feature_analysis.png")
print("\nWhat to look for:")
print("Plot 2: RSRP should decrease as distance increases")
print("Plot 3: Log distance should show a clear linear trend")
print("Plot 4: RSRP should decrease as angle off boresight increases")
print("Plot 5: Lower frequency bands should show higher mean RSRP")
print("Plot 6: Compare coverage quality across operators")

Saved: coverage_feature_analysis.png

What to look for:
Plot 2: RSRP should decrease as distance increases
Plot 3: Log distance should show a clear linear trend
Plot 4: RSRP should decrease as angle off boresight increases
Plot 5: Lower frequency bands should show higher mean RSRP
Plot 6: Compare coverage quality across operators


In [22]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time

print("=" * 55)
print("Project 3 — LTE Coverage Predictor")
print("Target: predict RSRP at unmeasured locations")
print("=" * 55)

# ── Step 1 — Select features ──────────────────────────────
feature_cols = [
    # Geometry — strongest predictors
    'distance_m',
    'log_distance',
    'angle_off_boresight',
    'height_m',

    # Signal measurements at this location
    'rsrq_db',
    'pathloss_db',
    'interference_ratio',

    # Frequency
    'freq_ghz',

    # Cell and operator
    'operator_num',

    # Time
    'hour',
    'is_peak',
]

target_col = 'rsrp_dbm'

# ── Step 2 — Clean for modeling ───────────────────────────
print("\nPreparing model dataset...")

df_model = df_work[feature_cols + [target_col]].dropna().copy()

print(f"Rows after dropping nulls: {len(df_model):,}")
print(f"Features: {len(feature_cols)}")
print(f"Target: {target_col}")

X = df_model[feature_cols]
y = df_model[target_col]

print(f"\nTarget variable stats:")
print(f"  Mean:   {y.mean():.2f} dBm")
print(f"  Std:    {y.std():.2f} dBm")
print(f"  Min:    {y.min():.2f} dBm")
print(f"  Max:    {y.max():.2f} dBm")

# ── Step 3 — Train test split ─────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"\nTraining samples: {len(X_train):,}")
print(f"Test samples:     {len(X_test):,}")

# ── Step 4 — Train Random Forest Regressor ────────────────
print("\nTraining Random Forest Regressor...")
print("(this may take 30-60 seconds on 450,000 rows)")

start = time.time()
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=10,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_time = time.time() - start
print(f"Training completed in {rf_time:.1f} seconds")

# ── Step 5 — Evaluate ─────────────────────────────────────
y_pred = rf_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

# Training accuracy — overfitting check
y_train_pred = rf_model.predict(X_train)
rmse_train   = np.sqrt(mean_squared_error(y_train, y_train_pred))
r2_train     = r2_score(y_train, y_train_pred)

print(f"\n=== Model Performance ===")
print(f"Test  RMSE:  {rmse:.2f} dBm")
print(f"Test  MAE:   {mae:.2f} dBm")
print(f"Test  R²:    {r2:.4f}")
print(f"\nTrain RMSE:  {rmse_train:.2f} dBm")
print(f"Train R²:    {r2_train:.4f}")
print(f"Overfitting gap: {(r2_train - r2):.4f}")

# ── Step 6 — Error distribution ───────────────────────────
errors = y_test - y_pred

print(f"\n=== Prediction Error Distribution ===")
print(f"Mean error:      {errors.mean():.2f} dBm  "
      f"(bias — positive = under-predicting)")
print(f"Std of errors:   {errors.std():.2f} dBm")
print(f"Within ±3 dBm:   "
      f"{(np.abs(errors) <= 3).sum()/len(errors)*100:.1f}%")
print(f"Within ±5 dBm:   "
      f"{(np.abs(errors) <= 5).sum()/len(errors)*100:.1f}%")
print(f"Within ±10 dBm:  "
      f"{(np.abs(errors) <= 10).sum()/len(errors)*100:.1f}%")

# ── Step 7 — Feature importance ───────────────────────────
print(f"\n=== Feature Importance ===")
importance = pd.Series(
    rf_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

for feat, imp in importance.items():
    bar = '█' * int(imp * 100)
    print(f"  {feat:<22} {imp:.3f} {bar}")

# ── Step 8 — Coverage quality accuracy ────────────────────
print(f"\n=== Coverage Classification Accuracy ===")
print(f"How accurately does the model identify")
print(f"good vs poor coverage areas?")

actual_good = (y_test > -85).astype(int)
pred_good   = (y_pred > -85).astype(int)
actual_poor = (y_test < -100).astype(int)
pred_poor   = (y_pred < -100).astype(int)

good_acc = (actual_good == pred_good).mean() * 100
poor_acc = (actual_poor == pred_poor).mean() * 100

print(f"  Good coverage detection (> -85 dBm): {good_acc:.1f}%")
print(f"  Poor coverage detection (< -100 dBm): {poor_acc:.1f}%")

Project 3 — LTE Coverage Predictor
Target: predict RSRP at unmeasured locations

Preparing model dataset...
Rows after dropping nulls: 566,452
Features: 11
Target: rsrp_dbm

Target variable stats:
  Mean:   -88.95 dBm
  Std:    10.71 dBm
  Min:    -123.30 dBm
  Max:    -48.40 dBm

Training samples: 453,161
Test samples:     113,291

Training Random Forest Regressor...
(this may take 30-60 seconds on 450,000 rows)
Training completed in 9.9 seconds

=== Model Performance ===
Test  RMSE:  1.91 dBm
Test  MAE:   1.37 dBm
Test  R²:    0.9683

Train RMSE:  1.70 dBm
Train R²:    0.9748
Overfitting gap: 0.0065

=== Prediction Error Distribution ===
Mean error:      -0.00 dBm  (bias — positive = under-predicting)
Std of errors:   1.91 dBm
Within ±3 dBm:   89.9%
Within ±5 dBm:   97.8%
Within ±10 dBm:  99.9%

=== Feature Importance ===
  pathloss_db            0.709 ██████████████████████████████████████████████████████████████████████
  distance_m             0.058 █████
  log_distance           

In [27]:
matched_cellids    = meas_cellids & all_cell_cellids
df_cells_matched   = df_all_cells[
    df_all_cells['cell_id'].isin(matched_cellids)
].drop_duplicates(subset=['cell_id']).copy()

print(f"Matched cells for heatmap: {len(df_cells_matched)}")

Matched cells for heatmap: 903


In [28]:
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

print("=" * 55)
print("Coverage Heatmap — Predicting RSRP across the city")
print("=" * 55)

# ── Step 1 — Define prediction grid ───────────────────────
# Create a grid of GPS points covering the drive test area
# Model predicts RSRP at every grid point

lat_min = df_work['latitude_meas'].min()
lat_max = df_work['latitude_meas'].max()
lon_min = df_work['longitude_meas'].min()
lon_max = df_work['longitude_meas'].max()

print(f"\nDrive test area:")
print(f"  Latitude:  {lat_min:.4f} to {lat_max:.4f}")
print(f"  Longitude: {lon_min:.4f} to {lon_max:.4f}")

# Grid resolution — 100x100 = 10,000 prediction points
grid_res = 100
lat_grid = np.linspace(lat_min, lat_max, grid_res)
lon_grid = np.linspace(lon_min, lon_max, grid_res)

# Create meshgrid — all combinations of lat and lon
lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)
lat_flat = lat_mesh.flatten()   # 10,000 points
lon_flat = lon_mesh.flatten()   # 10,000 points

print(f"\nPrediction grid: {grid_res}×{grid_res} = {grid_res**2:,} points")

# ── Step 2 — Assign best cell to each grid point ──────────
# For each grid point find the closest cell
# Use haversine distance to all 903 cells
# Assign the nearest cell's parameters

print("\nAssigning serving cell to each grid point...")

grid_features = []

for i, (lat, lon) in enumerate(zip(lat_flat, lon_flat)):
    # Distance from this grid point to all cells
    distances = haversine_distance(
        lat, lon,
        df_cells_matched['latitude'].values,
        df_cells_matched['longitude'].values
    )

    # Find nearest cell
    nearest_idx  = np.argmin(distances)
    nearest_cell = df_cells_matched.iloc[nearest_idx]
    nearest_dist = distances[nearest_idx]

    # Compute geometry features for this grid point
    bearing = compute_bearing(
        nearest_cell['latitude'], nearest_cell['longitude'],
        lat, lon
    )
    angle = np.abs(
        ((bearing - nearest_cell['azimuth_deg'] + 180) % 360) - 180
    )

    # Use network median values for non-geometric features
    grid_features.append({
        'distance_m':           nearest_dist,
        'log_distance':         np.log10(max(nearest_dist, 10)),
        'angle_off_boresight':  angle,
        'height_m':             nearest_cell['height_m'],
        'rsrq_db':              df_work['rsrq_db'].median(),
        'interference_ratio':   df_work['interference_ratio'].median(),
        'freq_ghz':             df_work['freq_ghz'].median(),
        'operator_num':         0,   # operator A
        'hour':                 12,  # midday
        'is_peak':              1,   # peak hours
    })

    if (i + 1) % 1000 == 0:
        print(f"  Processed {i+1:,}/{grid_res**2:,} grid points...")

df_grid = pd.DataFrame(grid_features)
print(f"Grid feature matrix: {df_grid.shape}")

# ── Step 3 — Predict RSRP at every grid point ─────────────
print("\nPredicting RSRP at all grid points...")

rsrp_predicted = rf_model_v2.predict(df_grid[feature_cols_v2])
rsrp_grid      = rsrp_predicted.reshape(grid_res, grid_res)

print(f"Predicted RSRP range:")
print(f"  Min: {rsrp_predicted.min():.1f} dBm")
print(f"  Max: {rsrp_predicted.max():.1f} dBm")
print(f"  Mean: {rsrp_predicted.mean():.1f} dBm")

# ── Step 4 — Coverage quality classification ──────────────
coverage_class = np.where(rsrp_grid > -85,  2,   # good
                 np.where(rsrp_grid > -100, 1,   # mid
                                            0))  # poor

good_pct = (coverage_class == 2).sum() / grid_res**2 * 100
mid_pct  = (coverage_class == 1).sum() / grid_res**2 * 100
poor_pct = (coverage_class == 0).sum() / grid_res**2 * 100

print(f"\nPredicted coverage quality:")
print(f"  Good (> -85 dBm):  {good_pct:.1f}%")
print(f"  Mid  (-100 to -85): {mid_pct:.1f}%")
print(f"  Poor (< -100 dBm): {poor_pct:.1f}%")

# ── Step 5 — Plot ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('LTE Coverage Prediction — Vienna Area',
             fontsize=14, fontweight='bold')

# ── Panel 1 — RSRP heatmap ────────────────────────────────
ax1 = axes[0]
rsrp_cmap = plt.cm.RdYlGn
rsrp_norm = mcolors.Normalize(vmin=-120, vmax=-50)

im1 = ax1.imshow(
    rsrp_grid,
    extent=[lon_min, lon_max, lat_min, lat_max],
    origin='lower',
    cmap=rsrp_cmap,
    norm=rsrp_norm,
    aspect='auto'
)
plt.colorbar(im1, ax=ax1, label='Predicted RSRP (dBm)')

# Overlay drive test route
ax1.scatter(
    df_work['longitude_meas'].sample(5000, random_state=42),
    df_work['latitude_meas'].sample(5000, random_state=42),
    c='white', s=0.3, alpha=0.3, label='Drive test route'
)

# Mark cell sites
ax1.scatter(
    df_cells_matched['longitude'],
    df_cells_matched['latitude'],
    c='blue', s=30, marker='^',
    zorder=5, label='Cell sites'
)

ax1.set_title('Predicted RSRP (dBm)', fontweight='bold')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.legend(fontsize=7, loc='upper right')

# ── Panel 2 — Coverage quality map ───────────────────────
ax2 = axes[1]
cmap_coverage = mcolors.ListedColormap(['#e74c3c', '#f39c12', '#2ecc71'])
bounds = [-0.5, 0.5, 1.5, 2.5]
norm_coverage = mcolors.BoundaryNorm(bounds, cmap_coverage.N)

im2 = ax2.imshow(
    coverage_class,
    extent=[lon_min, lon_max, lat_min, lat_max],
    origin='lower',
    cmap=cmap_coverage,
    norm=norm_coverage,
    aspect='auto'
)

ax2.scatter(
    df_work['longitude_meas'].sample(5000, random_state=42),
    df_work['latitude_meas'].sample(5000, random_state=42),
    c='white', s=0.3, alpha=0.2
)
ax2.scatter(
    df_cells_matched['longitude'],
    df_cells_matched['latitude'],
    c='blue', s=30, marker='^', zorder=5
)

legend_elements = [
    Patch(facecolor='#2ecc71', label=f'Good > -85 dBm ({good_pct:.0f}%)'),
    Patch(facecolor='#f39c12', label=f'Mid -100 to -85 ({mid_pct:.0f}%)'),
    Patch(facecolor='#e74c3c', label=f'Poor < -100 dBm ({poor_pct:.0f}%)')
]
ax2.legend(handles=legend_elements, fontsize=7, loc='upper right')
ax2.set_title('Coverage quality classification', fontweight='bold')
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')

# ── Panel 3 — Actual vs predicted scatter ────────────────
ax3 = axes[2]
sample = df_model.sample(10000, random_state=42)
X_sample  = sample[feature_cols_v2]
y_actual  = sample[target_col]
y_pred_s  = rf_model_v2.predict(X_sample)

ax3.scatter(y_actual, y_pred_s,
            alpha=0.1, s=1, color='steelblue')

# Perfect prediction line
lims = [-130, -45]
ax3.plot(lims, lims, 'r-', linewidth=1.5,
         label='Perfect prediction')
ax3.plot(lims, [l+5 for l in lims], 'r--',
         linewidth=0.8, alpha=0.5, label='±5 dBm')
ax3.plot(lims, [l-5 for l in lims], 'r--',
         linewidth=0.8, alpha=0.5)

ax3.set_xlim(lims)
ax3.set_ylim(lims)
ax3.set_title('Actual vs predicted RSRP', fontweight='bold')
ax3.set_xlabel('Actual RSRP (dBm)')
ax3.set_ylabel('Predicted RSRP (dBm)')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)
ax3.set_aspect('equal')

# Add R² annotation
ax3.text(0.05, 0.95,
         f'R² = {v2_r2:.4f}\nRMSE = {v2_rmse:.2f} dBm\nMAE = {v2_mae:.2f} dBm',
         transform=ax3.transAxes,
         fontsize=9,
         verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('coverage_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()

print("\nSaved: coverage_heatmap.png")
print("\nMap interpretation:")
print("  Green  — good coverage > -85 dBm")
print("  Orange — mid coverage -100 to -85 dBm")
print("  Red    — poor coverage < -100 dBm")
print("  Blue △ — cell site locations")
print("  White  — drive test route")

Coverage Heatmap — Predicting RSRP across the city

Drive test area:
  Latitude:  48.1583 to 48.2262
  Longitude: 16.2542 to 16.3982

Prediction grid: 100×100 = 10,000 points

Assigning serving cell to each grid point...
  Processed 1,000/10,000 grid points...
  Processed 2,000/10,000 grid points...
  Processed 3,000/10,000 grid points...
  Processed 4,000/10,000 grid points...
  Processed 5,000/10,000 grid points...
  Processed 6,000/10,000 grid points...
  Processed 7,000/10,000 grid points...
  Processed 8,000/10,000 grid points...
  Processed 9,000/10,000 grid points...
  Processed 10,000/10,000 grid points...
Grid feature matrix: (10000, 10)

Predicting RSRP at all grid points...
Predicted RSRP range:
  Min: -102.1 dBm
  Max: -66.5 dBm
  Mean: -93.7 dBm

Predicted coverage quality:
  Good (> -85 dBm):  9.7%
  Mid  (-100 to -85): 89.2%
  Poor (< -100 dBm): 1.1%

Saved: coverage_heatmap.png

Map interpretation:
  Green  — good coverage > -85 dBm
  Orange — mid coverage -100 to -85 d

In [29]:
import folium
from folium.plugins import HeatMap
import json

print("=" * 55)
print("Interactive Coverage Map — OpenStreetMap base layer")
print("=" * 55)

# ── Step 1 — Prepare heatmap data ─────────────────────────
# Folium HeatMap takes [lat, lon, weight] triplets
# Weight = intensity of color at that point
# We normalize RSRP to 0-1 scale for the heatmap

print("\nPreparing heatmap data...")

# Use actual measurements for the heatmap overlay
# Sample to keep file size manageable
df_map = df_work[['latitude_meas', 'longitude_meas',
                   'rsrp_dbm']].dropna().copy()

# Normalize RSRP to 0-1
# -50 dBm (best) → 1.0
# -120 dBm (worst) → 0.0
rsrp_min = -120
rsrp_max = -50
df_map['rsrp_norm'] = (
    (df_map['rsrp_dbm'] - rsrp_min) /
    (rsrp_max - rsrp_min)
).clip(0, 1)

# Sample for performance — 50,000 points renders well
df_map_sample = df_map.sample(50000, random_state=42)

# Format for folium HeatMap — [lat, lon, weight]
heat_data = df_map_sample[
    ['latitude_meas', 'longitude_meas', 'rsrp_norm']
].values.tolist()

print(f"Heatmap points: {len(heat_data):,}")

# ── Step 2 — Prepare predicted grid overlay ───────────────
# Use our model predictions for the grid
# Convert to circle markers colored by coverage quality

print("Preparing prediction grid overlay...")

# Coarser grid for the interactive map — 50x50 = 2500 points
grid_res_map = 50
lat_grid_map = np.linspace(lat_min, lat_max, grid_res_map)
lon_grid_map = np.linspace(lon_min, lon_max, grid_res_map)

grid_pred_features = []
grid_coords        = []

for lat in lat_grid_map:
    for lon in lon_grid_map:
        distances = haversine_distance(
            lat, lon,
            df_cells_matched['latitude'].values,
            df_cells_matched['longitude'].values
        )
        nearest_idx  = np.argmin(distances)
        nearest_cell = df_cells_matched.iloc[nearest_idx]
        nearest_dist = distances[nearest_idx]

        bearing = compute_bearing(
            nearest_cell['latitude'], nearest_cell['longitude'],
            lat, lon
        )
        angle = np.abs(
            ((bearing - nearest_cell['azimuth_deg'] + 180)
             % 360) - 180
        )

        grid_pred_features.append({
            'distance_m':          nearest_dist,
            'log_distance':        np.log10(max(nearest_dist, 10)),
            'angle_off_boresight': angle,
            'height_m':            nearest_cell['height_m'],
            'rsrq_db':             df_work['rsrq_db'].median(),
            'interference_ratio':  df_work['interference_ratio'].median(),
            'freq_ghz':            df_work['freq_ghz'].median(),
            'operator_num':        0,
            'hour':                12,
            'is_peak':             1,
        })
        grid_coords.append((lat, lon))

df_grid_map  = pd.DataFrame(grid_pred_features)
rsrp_grid_pred = rf_model_v2.predict(
    df_grid_map[feature_cols_v2]
)

print(f"Grid predictions: {len(rsrp_grid_pred):,} points")
print(f"RSRP range: {rsrp_grid_pred.min():.1f} to "
      f"{rsrp_grid_pred.max():.1f} dBm")

# ── Step 3 — Coverage color function ─────────────────────
def rsrp_to_color(rsrp):
    """Convert RSRP value to coverage color"""
    if rsrp > -85:
        return '#2ecc71'   # green — good
    elif rsrp > -95:
        return '#f1c40f'   # yellow — acceptable
    elif rsrp > -100:
        return '#e67e22'   # orange — poor
    else:
        return '#e74c3c'   # red — very poor

def rsrp_to_label(rsrp):
    """Convert RSRP to coverage quality label"""
    if rsrp > -85:
        return 'Good'
    elif rsrp > -95:
        return 'Acceptable'
    elif rsrp > -100:
        return 'Poor'
    else:
        return 'Very Poor'

# ── Step 4 — Build the folium map ─────────────────────────
print("\nBuilding interactive map...")

# Map center — center of drive test area
map_center = [
    (lat_min + lat_max) / 2,
    (lon_min + lon_max) / 2
]

m = folium.Map(
    location=map_center,
    zoom_start=13,
    tiles='OpenStreetMap'
)

# ── Layer 1 — Drive test heatmap ──────────────────────────
heatmap_layer = folium.FeatureGroup(name='Drive test heatmap')

HeatMap(
    heat_data,
    min_opacity=0.3,
    max_opacity=0.7,
    radius=8,
    blur=6,
    gradient={
        '0.0': '#e74c3c',   # red — poor
        '0.3': '#e67e22',   # orange
        '0.6': '#f1c40f',   # yellow
        '1.0': '#2ecc71'    # green — good
    }
).add_to(heatmap_layer)

heatmap_layer.add_to(m)

# ── Layer 2 — Predicted coverage grid ────────────────────
pred_layer = folium.FeatureGroup(
    name='Predicted coverage (ML model)',
    show=False   # hidden by default — toggle in layer control
)

for (lat, lon), rsrp in zip(grid_coords, rsrp_grid_pred):
    color = rsrp_to_color(rsrp)
    label = rsrp_to_label(rsrp)

    folium.CircleMarker(
        location=[lat, lon],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.5,
        opacity=0.5,
        popup=folium.Popup(
            f"<b>Predicted RSRP:</b> {rsrp:.1f} dBm<br>"
            f"<b>Coverage:</b> {label}<br>"
            f"<b>Location:</b> {lat:.4f}, {lon:.4f}",
            max_width=200
        ),
        tooltip=f"{rsrp:.1f} dBm — {label}"
    ).add_to(pred_layer)

pred_layer.add_to(m)

# ── Layer 3 — Cell sites ──────────────────────────────────
cell_layer = folium.FeatureGroup(name='Cell sites (903)')

for _, cell in df_cells_matched.iterrows():
    folium.Marker(
        location=[cell['latitude'], cell['longitude']],
        popup=folium.Popup(
            f"<b>Cell ID:</b> {cell['cell_id']}<br>"
            f"<b>PCI:</b> {cell['pci']}<br>"
            f"<b>Azimuth:</b> {cell['azimuth_deg']}°<br>"
            f"<b>Height:</b> {cell['height_m']}m<br>"
            f"<b>Operator:</b> {cell['operator']}",
            max_width=200
        ),
        tooltip=f"Cell {cell['cell_id']} — {cell['azimuth_deg']}°",
        icon=folium.Icon(
            color='blue',
            icon='signal',
            prefix='fa'
        )
    ).add_to(cell_layer)

cell_layer.add_to(m)

# ── Layer 4 — Poor coverage alerts ───────────────────────
alert_layer = folium.FeatureGroup(name='Poor coverage alerts')

poor_mask = rsrp_grid_pred < -100
poor_coords = [c for c, p in zip(grid_coords, poor_mask) if p]
poor_rsrps  = rsrp_grid_pred[poor_mask]

for (lat, lon), rsrp in zip(poor_coords, poor_rsrps):
    folium.CircleMarker(
        location=[lat, lon],
        radius=10,
        color='#e74c3c',
        fill=True,
        fill_color='#e74c3c',
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>COVERAGE ALERT</b><br>"
            f"<b>Predicted RSRP:</b> {rsrp:.1f} dBm<br>"
            f"<b>Status:</b> Poor coverage hole<br>"
            f"<b>Action:</b> Field investigation needed",
            max_width=220
        ),
        tooltip=f"ALERT: {rsrp:.1f} dBm"
    ).add_to(alert_layer)

alert_layer.add_to(m)

# ── Layer control — toggle layers on/off ─────────────────
folium.LayerControl(collapsed=False).add_to(m)

# ── Legend ────────────────────────────────────────────────
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px;
     z-index: 1000; background-color: white;
     padding: 10px; border-radius: 8px;
     border: 2px solid grey; font-size: 12px;">
<b>Coverage Quality</b><br>
<i style="background:#2ecc71;
   width:12px;height:12px;
   display:inline-block;border-radius:50%"></i>
&nbsp;Good (&gt; -85 dBm)<br>
<i style="background:#f1c40f;
   width:12px;height:12px;
   display:inline-block;border-radius:50%"></i>
&nbsp;Acceptable (-85 to -95 dBm)<br>
<i style="background:#e67e22;
   width:12px;height:12px;
   display:inline-block;border-radius:50%"></i>
&nbsp;Poor (-95 to -100 dBm)<br>
<i style="background:#e74c3c;
   width:12px;height:12px;
   display:inline-block;border-radius:50%"></i>
&nbsp;Very Poor (&lt; -100 dBm)<br>
<hr style="margin:5px 0">
<b>Model:</b> Random Forest Regressor<br>
<b>RMSE:</b> 4.50 dBm &nbsp;
<b>R²:</b> 0.8239<br>
<b>Data:</b> 567,195 LTE measurements
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# ── Save ──────────────────────────────────────────────────
output_path = r'C:\Users\SujitR\Desktop\rfai\coverage_heatmap.html'
m.save(output_path)

print(f"\nInteractive map saved: coverage_heatmap.html")
print(f"\nMap layers:")
print(f"  1. Drive test heatmap    — actual measurements")
print(f"  2. Predicted coverage    — ML model output")
print(f"  3. Cell sites            — 903 cells with parameters")
print(f"  4. Poor coverage alerts  — {len(poor_coords)} flagged locations")
print(f"\nOpen coverage_heatmap.html in any browser")
print(f"Toggle layers using the control panel top right")
print(f"Click any marker for details")
print(f"Click any predicted grid point for RSRP value")

Interactive Coverage Map — OpenStreetMap base layer

Preparing heatmap data...
Heatmap points: 50,000
Preparing prediction grid overlay...
Grid predictions: 2,500 points
RSRP range: -101.4 to -69.8 dBm

Building interactive map...

Interactive map saved: coverage_heatmap.html

Map layers:
  1. Drive test heatmap    — actual measurements
  2. Predicted coverage    — ML model output
  3. Cell sites            — 903 cells with parameters
  4. Poor coverage alerts  — 25 flagged locations

Open coverage_heatmap.html in any browser
Toggle layers using the control panel top right
Click any marker for details
Click any predicted grid point for RSRP value
